# Two specialized agents

This lab gives the same fictional case to two agents with different responsibilities. Their answers are displayed separately; they do not yet hand work to one another.

## Step 1: Import AgentScope and configuration helpers

This cell imports the agent, message, model, and local-configuration classes used throughout the lab.

In [ ]:
import os

from dotenv import load_dotenv
from agentscope.agent import Agent, ReActConfig
from agentscope.credential import OpenAICredential
from agentscope.message import Msg, TextBlock
from agentscope.model import OpenAIChatModel


## Step 2: Configure the shared model connection

Both specialists use this one model object. Their prompts and conversation states remain separate.

In [ ]:
load_dotenv()
model_name = os.getenv("MODEL")
base_url = os.getenv("OLLAMA_BASE_URL")

if not model_name or not base_url:
    raise RuntimeError("Set MODEL and OLLAMA_BASE_URL in .env before running this notebook.")

model = OpenAIChatModel(
    credential=OpenAICredential(api_key="ollama", base_url=base_url),
    model=model_name,
    stream=False,
    parameters=OpenAIChatModel.Parameters(temperature=0, max_tokens=160),
)


## Step 3: Create agents with non-overlapping responsibilities

The network specialist describes what the alert observed and what it cannot show. The evidence specialist identifies supported facts and missing evidence.

In [ ]:
network_specialist = Agent(
    name="network_specialist",
    system_prompt=(
        "You are the network specialist for a practice security case. "
        "Report what the network alert observed and what the alert cannot show."
    ),
    model=model,
    react_config=ReActConfig(max_iters=2),
)

evidence_specialist = Agent(
    name="evidence_specialist",
    system_prompt=(
        "You are the evidence specialist for a practice security case. "
        "List supported facts, missing evidence, and one next item to collect."
    ),
    model=model,
    react_config=ReActConfig(max_iters=2),
)

print(f"Created: {network_specialist.name} and {evidence_specialist.name}")


## Step 4: Define the shared fictional case

This is the entire evidence set available to both agents. The alert is an automated network-monitoring record: it establishes that a connection pattern was observed, not why it happened or whether it was harmful. Keeping the materials fixed makes the specialist outputs easier to compare.

In [ ]:
case_materials = """
Practice case INC-204
- Network-monitoring alert: at 09:14 UTC, an automated sensor recorded a workstation making forty-three outbound contacts to 192.0.2.44. The alert supplies the time, the workstation, the destination IP address, and the contact count.
- The local practice list marks 192.0.2.44 as suspicious and says this requires analyst review; it is not proof of malicious activity.
- This alert contains connection metadata only. It does not identify the process that made the contacts, the user action that caused them, the transmitted payload (the data sent), or the destination port/service.
""".strip()

question = (
    "Review the following practice case. Give two concise bullets and clearly state uncertainty.\n\n"
    f"{case_materials}"
)
print(case_materials)


## Step 5: Ask each specialist independently

Each call receives the same user message. Calling `reply()` on one agent does not send its answer to the other agent.

In [ ]:
shared_request = Msg(
    name="analyst",
    role="user",
    content=[TextBlock(text=question)],
)

network_response = await network_specialist.reply(shared_request)
evidence_response = await evidence_specialist.reply(shared_request)


## Step 6: Compare the separate findings

This cell extracts text from each response and prints it under the responsible specialist's label. Read the answers for scope: connection details belong to the network specialist, while evidence gaps belong to the evidence specialist.

In [ ]:
def response_text(response: Msg) -> str:
    """Extract readable text from an AgentScope response message."""
    return "".join(block.text for block in response.content if isinstance(block, TextBlock))

print("NETWORK SPECIALIST:")
print(response_text(network_response))
print("\nEVIDENCE SPECIALIST:")
print(response_text(evidence_response))


## Step 7: Checkpoint

Add a third agent that proposes only the next evidence to collect. Keep the existing two prompts unchanged, then verify that no agent calls the activity malicious from these limited practice facts.